In [35]:
import sys
sys.path.append('..')
import torch
import soundfile as sf
import json
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import argparse
import random

from encoder import FMEncoder, compute_spectrogram_cqt
from fm_ddsp_batched import fm_renderer_batched, make_mod_matrix_batched
from nnAudio.features import CQT2010v2
from generate_dataset import generate_dataset
from train_batch import train_stage1

%load_ext autoreload
%autoreload 2

# Stage 1 — Single modulator, single carrier
    # Fixed f0=440, one modulator → one carrier, vary ratio of modulator and level. Encoder learns the relationship between ratio and sideband position.
# Stage 2 — Two operators, fixed algorithm
    # Still fixed f0, introduce a second modulator. Encoder learns to handle multiple simultaneous modulators.
# Stage 3 — Variable f0, fixed algorithm
    # Now vary f0 across MIDI range. Encoder learns pitch invariance — same timbre at different pitches should give same parameters.
# Stage 4 — Multiple algorithms
    # Introduce algorithm variety. Encoder learns to distinguish different routing topologies.
# Stage 5 — Full dataset
    # Everything varying simultaneously.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
def stage1_params():
    ratio_choices = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
    # only use operator 2
    ratio_op2 = random.choice(ratio_choices)
    mod_level_op2 = torch.rand(1)
    return {
        'f0': 440.0,
        'algorithm': 'algo_2',
        'mod_values': torch.tensor([0,0,0,0,0,0,1], dtype=torch.float32),
        'ratios': torch.tensor([1,1,ratio_op2,1], dtype = torch.float32),
        'levels': torch.tensor([0,0,mod_level_op2, 1], dtype = torch.float32),
        'carrier_weights': torch.tensor([0.0, 0.0, 0.0, 1.0], dtype = torch.float32)
    }

In [37]:
save_dir_pc = "B:\\TrainingData\\stage1"
#save_dir = '../data/stage1',
args = argparse.Namespace(
    n_examples = 40000,
    save_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    seed = 127,
    overwrite=True
)
#generate_dataset(args, param_fn = stage1_params)

In [ ]:
# train stage 1
args = argparse.Namespace(
    data_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    lr = 0.0001,
    n_epochs = 100,
    batch_size = 32,
    resume = None,
    start_epoch = 0
)
train_stage1(args)

Using Cuda? True
Low pass filter created, time used = 0.0000 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0020 seconds
Epoch 0/100:
Average epoch loss: 0.22370925518274307
Average weight change: 0.016859598457813263
Epoch 1/100:
Average epoch loss: 0.1637275095462799
Average weight change: 0.005992631893604994
Epoch 2/100:
Average epoch loss: 0.16087876073122023
Average weight change: 0.0035227707121521235
Epoch 3/100:
Average epoch loss: 0.16250689099431037
Average weight change: 0.0022552392911165953
Epoch 4/100:
